In [ ]:
# DEPTH-CONDITIONAL EVALUATION OF EXPLICIT COMORBIDITY
# INTERACTION MODELING IN MULTI-LABEL DIABETIC COMPLICATION
# PREDICTION
#
# Automation Notebook — Depth Experiment (100% Data)
# Models: Baseline vs Linear Additive
# Depths: 1, 2, 3 Hidden Layers



import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, hamming_loss, accuracy_score, roc_curve
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import random
import os
import json
import shutil
from tqdm.notebook import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:

from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# Load dataset and assign column names
col_names = ['ID', 'AGE', 'SEX', 'INF_ANAM', 'STENOK_AN', 'FK_STENOK', 'IBS_POST', 'IBS_NASL', 'GB', 'SIM_GIPERT', 'DLIT_AG', 'ZSN_A', 'nr11', 'nr01', 'nr02', 'nr03', 'nr04', 'nr07', 'nr08', 'np01', 'np04', 'np05', 'np07', 'np08', 'np09', 'np10', 'endocr_01', 'endocr_02', 'endocr_03', 'zab_leg_01', 'zab_leg_02', 'zab_leg_03', 'zab_leg_04', 'zab_leg_06', 'S_AD_KBRIG', 'D_AD_KBRIG', 'S_AD_ORIT', 'D_AD_ORIT', 'O_L_POST', 'K_SH_POST', 'MP_TP_POST', 'SVT_POST', 'GT_POST', 'FIB_G_POST', 'ant_im', 'lat_im', 'inf_im', 'post_im', 'IM_PG_P', 'ritm_ecg_p_01', 'ritm_ecg_p_02', 'ritm_ecg_p_04', 'ritm_ecg_p_06', 'ritm_ecg_p_07', 'ritm_ecg_p_08', 'n_r_ecg_p_01', 'n_r_ecg_p_02', 'n_r_ecg_p_03', 'n_r_ecg_p_04', 'n_r_ecg_p_05', 'n_r_ecg_p_06', 'n_r_ecg_p_08', 'n_r_ecg_p_09', 'n_r_ecg_p_10', 'n_p_ecg_p_01', 'n_p_ecg_p_03', 'n_p_ecg_p_04', 'n_p_ecg_p_05', 'n_p_ecg_p_06', 'n_p_ecg_p_07', 'n_p_ecg_p_08', 'n_p_ecg_p_09', 'n_p_ecg_p_10', 'n_p_ecg_p_11', 'n_p_ecg_p_12', 'fibr_ter_01', 'fibr_ter_02', 'fibr_ter_03', 'fibr_ter_05', 'fibr_ter_06', 'fibr_ter_07', 'fibr_ter_08', 'GIPO_K', 'K_BLOOD', 'GIPER_NA', 'Na_BLOOD', 'ALT_BLOOD', 'AST_BLOOD', 'KFK_BLOOD', 'L_BLOOD', 'ROE', 'TIME_B_S', 'R_AB_1_n', 'R_AB_2_n', 'R_AB_3_n', 'NA_KB', 'NOT_NA_KB', 'LID_KB', 'NITR_S', 'NA_R_1_n', 'NA_R_2_n', 'NA_R_3_n', 'NOT_NA_1_n', 'NOT_NA_2_n', 'NOT_NA_3_n', 'LID_S_n', 'B_BLOK_S_n', 'ANT_CA_S_n', 'GEPAR_S_n', 'ASP_S_n', 'TIKL_S_n', 'TRENT_S_n', 'FIBR_PREDS', 'PREDS_TAH', 'JELUD_TAH', 'FIBR_JELUD', 'A_V_BLOK', 'OTEK_LANC', 'RAZRIV', 'DRESSLER', 'ZSN', 'REC_IM', 'P_IM_STEN', 'LET_IS']
df = pd.read_csv('../../data/MI.data', header=None, names=col_names, na_values='?')
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")

# ── Feature Renaming Map ───────────────────────────────────────────────────────
feature_rename = {
    'AGE':          'age', 'SEX':          'sex', 'INF_ANAM':     'prev_mi_count',
    'STENOK_AN':    'angina_history', 'FK_STENOK':    'angina_functional_class',
    'IBS_POST':     'coronary_disease_history', 'IBS_NASL':     'hereditary_coronary_disease',
    'GB':           'hypertension_stage', 'SIM_GIPERT':   'symptomatic_hypertension',
    'DLIT_AG':      'hypertension_duration_years', 'ZSN_A':        'chronic_heart_failure_history',
    'nr11':         'neuro_disorder_cerebrovascular', 'nr01':         'neuro_disorder_01',
    'nr02':         'neuro_disorder_02', 'nr03':         'neuro_disorder_03',
    'nr04':         'neuro_disorder_04', 'nr07':         'neuro_disorder_07',
    'nr08':         'neuro_disorder_08', 'np01':         'neuro_path_01',
    'np04':         'neuro_path_04', 'np05':         'neuro_path_05',
    'np07':         'neuro_path_07', 'np08':         'neuro_path_08',
    'np09':         'neuro_path_09', 'np10':         'neuro_path_10',
    'endocr_01':    'diabetes', 'endocr_02':    'obesity', 'endocr_03':    'thyroid_disorder',
    'zab_leg_01':   'chronic_bronchitis', 'zab_leg_02':   'obstructive_bronchitis',
    'zab_leg_03':   'bronchial_asthma', 'zab_leg_04':   'chronic_pneumonia',
    'zab_leg_06':   'pulmonary_tuberculosis', 'S_AD_KBRIG':   'systolic_bp_ambulance',
    'D_AD_KBRIG':   'diastolic_bp_ambulance', 'S_AD_ORIT':    'systolic_bp_icu',
    'D_AD_ORIT':    'diastolic_bp_icu', 'O_L_POST':     'pulmonary_edema_at_onset',
    'K_SH_POST':    'cardiogenic_shock_at_onset', 'MP_TP_POST':   'atrial_fibrillation_at_onset',
    'SVT_POST':     'supraventricular_tachycardia_at_onset', 'GT_POST':      'ventricular_tachycardia_at_onset',
    'FIB_G_POST':   'ventricular_fibrillation_at_onset', 'ant_im':       'infarction_anterior',
    'lat_im':       'infarction_lateral', 'inf_im':       'infarction_inferior',
    'post_im':      'infarction_posterior', 'IM_PG_P':      'infarction_right_ventricle',
    'ritm_ecg_p_01': 'ecg_rhythm_sinus_normal', 'ritm_ecg_p_02': 'ecg_rhythm_sinus_bradycardia',
    'ritm_ecg_p_04': 'ecg_rhythm_atrial_fibrillation', 'ritm_ecg_p_06': 'ecg_rhythm_ventricular_pacemaker',
    'ritm_ecg_p_07': 'ecg_rhythm_supraventricular_tachycardia', 'ritm_ecg_p_08': 'ecg_rhythm_sinus_tachycardia',
    'n_r_ecg_p_01': 'ecg_no_rhythm_changes', 'n_r_ecg_p_02': 'ecg_premature_atrial_beats',
    'n_r_ecg_p_03': 'ecg_frequent_premature_atrial', 'n_r_ecg_p_04': 'ecg_premature_ventricular_beats',
    'n_r_ecg_p_05': 'ecg_frequent_premature_ventricular', 'n_r_ecg_p_06': 'ecg_bigeminy',
    'n_r_ecg_p_08': 'ecg_ventricular_fibrillation_rhythm', 'n_r_ecg_p_09': 'ecg_supraventricular_tachycardia_rhythm',
    'n_r_ecg_p_10': 'ecg_paroxysmal_ventricular_tachycardia', 'n_p_ecg_p_01': 'ecg_no_conduction_changes',
    'n_p_ecg_p_03': 'ecg_lbbb', 'n_p_ecg_p_04': 'ecg_rbbb', 'n_p_ecg_p_05': 'ecg_lbbb_rbbb_combined',
    'n_p_ecg_p_06': 'ecg_first_degree_av_block', 'n_p_ecg_p_07': 'ecg_second_degree_av_block_type1',
    'n_p_ecg_p_08': 'ecg_second_degree_av_block_type2', 'n_p_ecg_p_09': 'ecg_third_degree_av_block',
    'n_p_ecg_p_10': 'ecg_sinoatrial_block', 'n_p_ecg_p_11': 'ecg_incomplete_lbbb',
    'n_p_ecg_p_12': 'ecg_incomplete_rbbb', 'fibr_ter_01':  'thrombolysis_streptokinase',
    'fibr_ter_02':  'thrombolysis_tpa', 'fibr_ter_03':  'thrombolysis_alteplase',
    'fibr_ter_05':  'thrombolysis_streptodecase', 'fibr_ter_06':  'thrombolysis_urokinase',
    'fibr_ter_07':  'thrombolysis_bilignosin', 'fibr_ter_08':  'thrombolysis_tenecteplase',
    'GIPO_K':       'hypokalemia', 'K_BLOOD':      'potassium_blood',
    'GIPER_NA':     'hypernatremia', 'Na_BLOOD':     'sodium_blood',
    'ALT_BLOOD':    'alt_blood', 'AST_BLOOD':    'ast_blood',
    'KFK_BLOOD':    'creatine_kinase_blood', 'L_BLOOD':      'wbc_count',
    'ROE':          'esr_inflammation', 'TIME_B_S':     'time_onset_to_hospital_hours',
    'R_AB_1_n':     'antiarrhythmic_day1', 'R_AB_2_n':     'antiarrhythmic_day2',
    'R_AB_3_n':     'antiarrhythmic_day3', 'NA_KB':        'lidocaine_ambulance',
    'NOT_NA_KB':    'other_antiarrhythmic_ambulance', 'LID_KB':       'lidocaine_hospital',
    'NITR_S':       'nitrates', 'NA_R_1_n':     'lidocaine_day1',
    'NA_R_2_n':     'lidocaine_day2', 'NA_R_3_n':     'lidocaine_day3',
    'NOT_NA_1_n':   'other_antiarrhythmic_day1', 'NOT_NA_2_n':   'other_antiarrhythmic_day2',
    'NOT_NA_3_n':   'other_antiarrhythmic_day3', 'LID_S_n':      'lidocaine_infusion',
    'B_BLOK_S_n':   'beta_blockers', 'ANT_CA_S_n':   'calcium_antagonists',
    'GEPAR_S_n':    'heparin', 'ASP_S_n':      'aspirin',
    'TIKL_S_n':     'ticlopidine', 'TRENT_S_n':    'trental',
}

target_rename = {
    'FIBR_PREDS':   'atrial_fibrillation', 'PREDS_TAH':    'supraventricular_tachycardia',
    'JELUD_TAH':    'ventricular_tachycardia', 'FIBR_JELUD':   'ventricular_fibrillation',
    'A_V_BLOK':     'av_block', 'OTEK_LANC':    'pulmonary_edema',
    'RAZRIV':       'myocardial_rupture', 'DRESSLER':     'dressler_syndrome',
    'ZSN':          'heart_failure', 'REC_IM':       'recurrent_mi',
    'P_IM_STEN':    'postinfarction_angina', 'LET_IS':       'lethal_outcome',
}

df = df.rename(columns={**feature_rename, **target_rename})
if 'ID' in df.columns:
    df = df.drop(columns=['ID'])

# ── 1. Define the 3 ventricular/infranodal complication targets ────────────
complications = [
    'ventricular_tachycardia',
    'ventricular_fibrillation',
    'av_block',
]

EXCLUDED_TARGETS = [
    'myocardial_rupture', 'dressler_syndrome', 'postinfarction_angina', 'lethal_outcome',
    'pulmonary_edema', 'heart_failure', 'recurrent_mi',
]

LEAKAGE_COLUMNS = [
    'atrial_fibrillation_at_onset', 'supraventricular_tachycardia_at_onset',
    'ventricular_tachycardia_at_onset', 'ventricular_fibrillation_at_onset',
    'pulmonary_edema_at_onset', 'cardiogenic_shock_at_onset',
    'ecg_rhythm_sinus_normal', 'ecg_rhythm_sinus_bradycardia',
    'ecg_rhythm_atrial_fibrillation', 'ecg_rhythm_ventricular_pacemaker',
    'ecg_rhythm_supraventricular_tachycardia', 'ecg_rhythm_sinus_tachycardia',
    'ecg_no_rhythm_changes', 'ecg_premature_atrial_beats',
    'ecg_frequent_premature_atrial', 'ecg_premature_ventricular_beats',
    'ecg_frequent_premature_ventricular', 'ecg_bigeminy',
    'ecg_ventricular_fibrillation_rhythm', 'ecg_supraventricular_tachycardia_rhythm',
    'ecg_paroxysmal_ventricular_tachycardia', 'ecg_no_conduction_changes',
    'ecg_lbbb', 'ecg_rbbb', 'ecg_lbbb_rbbb_combined', 'ecg_first_degree_av_block',
    'ecg_second_degree_av_block_type1', 'ecg_second_degree_av_block_type2',
    'ecg_third_degree_av_block', 'ecg_sinoatrial_block', 'ecg_incomplete_lbbb',
    'ecg_incomplete_rbbb',
]

exclude_cols = set(EXCLUDED_TARGETS + LEAKAGE_COLUMNS + complications +
                   ['atrial_fibrillation', 'supraventricular_tachycardia'])

missingness  = df.isnull().mean()
high_missing = [c for c in df.columns if missingness[c] > 0.50 and c not in exclude_cols]
exclude_cols.update(high_missing)

feature_cols = [c for c in df.columns if c not in exclude_cols]

X_df = df[feature_cols].copy()
y_df = df[complications].copy()

y_df = y_df.fillna(0).astype(np.float32)

feature_imputer = SimpleImputer(strategy='median')
X_imputed       = feature_imputer.fit_transform(X_df)

X = X_imputed.astype(np.float32)
y = y_df.values.astype(np.float32)

# ── 6. Stratified train / val / test split ─────────────────────────────────
y_combined   = y.dot(2**np.arange(y.shape[1]))
combo_counts = pd.Series(y_combined).value_counts()
can_stratify = (combo_counts >= 2).all()

from sklearn.model_selection import train_test_split
if can_stratify:
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y_combined
    )
    y_temp_combined = y_temp.dot(2**np.arange(y_temp.shape[1]))
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp_combined
    )
else:
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42)

from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import DataLoader, TensorDataset

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

batch_size   = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=batch_size, shuffle=False)

input_dim = X_train.shape[1]
print(f"   Input dim  : {input_dim}")
print(f"   Output dim : {len(complications)} labels (VT, VF, AV Block)")



In [ ]:















class SharedBackbone(nn.Module):
    """
    Constrained shared backbone.
    hidden_dim=6, proj_dim=8, n_hidden_layers configurable.
    """
    def __init__(self, input_dim, hidden_dim=6, n_hidden_layers=1, proj_dim=8):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers += [
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2),
            ]
            in_dim = hidden_dim
        layers += [
            nn.Linear(hidden_dim, proj_dim),
            nn.BatchNorm1d(proj_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        ]
        self.net        = nn.Sequential(*layers)
        self.output_dim = proj_dim

    def forward(self, x):
        return self.net(x)


class BaselineModel(nn.Module):
    """Shared backbone + independent linear head. No interaction."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head     = nn.Linear(backbone.output_dim, 3)

    def forward(self, x):
        return self.head(self.backbone(x))


class LinearAdditiveHead(nn.Module):
    """
    Linear additive interaction head.
    Diagonal of A is permanently masked — receives no gradients.
    """
    def __init__(self, hidden_dim, n_labels=3, init_scale=0.01):
        super().__init__()
        self.base = nn.Linear(hidden_dim, n_labels)
        self.A    = nn.Parameter(torch.zeros(n_labels, n_labels))
        with torch.no_grad():
            self.A.data.fill_(init_scale)
            self.A.data.fill_diagonal_(0)
        mask = 1 - torch.eye(n_labels)
        self.register_buffer('mask', mask)

    def forward(self, h):
        base_logits = self.base(h)
        base_probs  = torch.sigmoid(base_logits)
        A_masked    = self.A * self.mask
        interaction = base_probs @ A_masked.T
        return base_logits + interaction


class LinearAdditiveModel(nn.Module):
    """Shared backbone + linear additive interaction head."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head     = LinearAdditiveHead(backbone.output_dim, n_labels=3)

    def forward(self, x):
        return self.head(self.backbone(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Verify parameter budget
bb   = SharedBackbone(input_dim)
base = BaselineModel(bb)
n_params = count_params(base)
print(f"Architecture: hidden_dim=6, proj_dim=8, n_hidden_layers=1")
print(f"Total params:    {n_params:,}")
print(f"Samples/param:   {len(X_train)/n_params:.2f}")
print("\n Model definitions ready.")


In [ ]:














def train_model(model, train_loader, val_loader, model_name,
                epochs=300, lr=0.001, patience=40):
    model     = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=15, factor=0.5
    )

    train_losses     = []
    val_aurocs       = []
    best_val_auroc   = 0
    patience_counter = 0
    best_state       = None

    pbar = tqdm(range(epochs), desc=f'{model_name}', unit='epoch',
                bar_format='{l_bar}{bar:30}{r_bar}')

    for epoch in pbar:
        # ── Train ─────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        epoch_loss /= len(train_loader)
        train_losses.append(epoch_loss)

        # ── Validate ──────────────────────────────────────────────────────
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                all_logits.append(model(X_batch.to(device)).cpu())
                all_labels.append(y_batch.cpu())

        all_probs  = torch.sigmoid(torch.cat(all_logits))
        all_labels = torch.cat(all_labels)
        val_auroc  = roc_auc_score(all_labels.numpy(), all_probs.numpy(), average='macro')
        val_aurocs.append(val_auroc)

        scheduler.step(val_auroc)
        lr_now = optimizer.param_groups[0]['lr']

        pbar.set_postfix({
            'loss':      f'{epoch_loss:.4f}',
            'val_auroc': f'{val_auroc:.4f}',
            'best':      f'{best_val_auroc:.4f}',
            'lr':        f'{lr_now:.6f}',
            'patience':  f'{patience_counter}/{patience}'
        })

        if val_auroc > best_val_auroc:
            best_val_auroc   = val_auroc
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                pbar.set_description(f'{model_name} [EARLY STOP ep={epoch+1}]')
                break

    model.load_state_dict(best_state)
    return model, train_losses, val_aurocs, best_val_auroc


print(" Training function ready.")

In [ ]:

def evaluate_model(model, test_loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            all_logits.append(model(X_batch.to(device)).cpu())
            all_labels.append(y_batch)

    probs  = torch.sigmoid(torch.cat(all_logits)).numpy()
    labels = torch.cat(all_labels).numpy()
    preds  = (probs > 0.5).astype(int)

    per_label_auroc = {
        comp: roc_auc_score(labels[:, i], probs[:, i])
        for i, comp in enumerate(complications)
    }

    return {
        'auroc_macro':      roc_auc_score(labels, probs, average='macro'),
        'per_label_auroc':  per_label_auroc,
        'f1_macro':         f1_score(labels, preds, average='macro'),
        'hamming_loss':     hamming_loss(labels, preds),
        'subset_accuracy':  accuracy_score(labels, preds),
        'probabilities':    probs,
        'labels':           labels,
    }


print(" Evaluation function ready.")

In [ ]:

COMP_FULL = ['VT', 'VF', 'AV Block']
C_BASE    = '#2563EB'
C_LIN     = '#DC2626'

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.color':        '#E5E7EB',
    'grid.linewidth':    0.6,
    'axes.labelsize':    11,
    'axes.titlesize':    12,
    'axes.titleweight':  'bold',
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,
    'legend.framealpha': 0.9,
    'figure.dpi':        150,
})


def save_all_plots(n_hidden, SAVE_DIR,
                   baseline_losses, linear_losses,
                   baseline_aurocs, linear_aurocs,
                   baseline_best_val, linear_best_val,
                   baseline_results, linear_results,
                   A_matrix):

    depth_str = f'{n_hidden} Hidden Layer{"s" if n_hidden > 1 else ""}'

    # ── PLOT 1: TRAINING DYNAMICS ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Training Dynamics — {depth_str}',
                 fontsize=14, fontweight='bold', y=1.01)

    ax = axes[0]
    ax.plot(baseline_losses, color=C_BASE, linewidth=2, label='Baseline')
    ax.plot(linear_losses,   color=C_LIN,  linewidth=2, label='Linear Additive')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Training Loss (BCE)')
    ax.set_title('Training Loss')
    ax.legend()

    ax = axes[1]
    ax.plot(baseline_aurocs, color=C_BASE, linewidth=2,
            label=f'Baseline (best = {baseline_best_val:.4f})')
    ax.plot(linear_aurocs,   color=C_LIN,  linewidth=2,
            label=f'Linear Additive (best = {linear_best_val:.4f})')
    ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1,
               alpha=0.5, label='Random baseline (0.5)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation AUROC (macro)')
    ax.set_title('Validation AUROC')
    ax.set_ylim(bottom=0.45)
    ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'plot1_training_dynamics.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("   Plot 1: Training dynamics")

    # ── PLOT 2: METRIC COMPARISON ─────────────────────────────────────────
    metrics = {
        'AUROC\n(macro)':    (baseline_results['auroc_macro'],     linear_results['auroc_macro']),
        'F1\n(macro)':       (baseline_results['f1_macro'],        linear_results['f1_macro']),
        'Subset\nAccuracy':  (baseline_results['subset_accuracy'], linear_results['subset_accuracy']),
        'Hamming\nLoss (↓)': (baseline_results['hamming_loss'],    linear_results['hamming_loss']),
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Model Performance Comparison — {depth_str}',
                 fontsize=14, fontweight='bold', y=1.01)

    x           = np.arange(len(metrics))
    width       = 0.35
    labels_list = list(metrics.keys())
    base_vals   = [v[0] for v in metrics.values()]
    linear_vals = [v[1] for v in metrics.values()]

    ax     = axes[0]
    bars_b = ax.bar(x - width/2, base_vals,   width, label='Baseline',        color=C_BASE, alpha=0.85)
    bars_l = ax.bar(x + width/2, linear_vals, width, label='Linear Additive', color=C_LIN,  alpha=0.85)
    for bar in bars_b:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{bar.get_height():.4f}', ha='center', va='bottom',
                fontsize=8.5, color=C_BASE, fontweight='bold')
    for bar in bars_l:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{bar.get_height():.4f}', ha='center', va='bottom',
                fontsize=8.5, color=C_LIN,  fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels_list)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.08)
    ax.legend()
    ax.set_title('All Metrics')

    ax     = axes[1]
    diffs  = [l - b for b, l in zip(base_vals, linear_vals)]
    colors = [C_LIN if d >= 0 else C_BASE for d in diffs]
    hamming_idx = labels_list.index('Hamming\nLoss (↓)')
    colors[hamming_idx] = C_LIN if diffs[hamming_idx] <= 0 else C_BASE
    bars = ax.bar(x, diffs, width=0.5, color=colors, alpha=0.85)
    ax.axhline(y=0, color='black', linewidth=0.8)
    for bar, d in zip(bars, diffs):
        ypos = bar.get_height() + 0.0005 if d >= 0 else bar.get_height() - 0.002
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'{d:+.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels_list)
    ax.set_ylabel('Δ (Linear Additive − Baseline)')
    ax.set_title('Performance Gap')

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'plot2_metric_comparison.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("   Plot 2: Metric comparison")

    # ── PLOT 3: PER-LABEL AUROC + ROC CURVES ─────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Per-Label Analysis — {depth_str}',
                 fontsize=14, fontweight='bold', y=1.01)

    ax     = axes[0]
    base_p = [baseline_results['per_label_auroc'][c] for c in complications]
    lin_p  = [linear_results['per_label_auroc'][c]   for c in complications]
    x      = np.arange(3)
    width  = 0.35
    bars_b = ax.bar(x - width/2, base_p, width, label='Baseline',        color=C_BASE, alpha=0.85)
    bars_l = ax.bar(x + width/2, lin_p,  width, label='Linear Additive', color=C_LIN,  alpha=0.85)
    for bar in bars_b:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{bar.get_height():.4f}', ha='center', va='bottom',
                fontsize=9, color=C_BASE, fontweight='bold')
    for bar in bars_l:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{bar.get_height():.4f}', ha='center', va='bottom',
                fontsize=9, color=C_LIN,  fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(COMP_FULL)
    ax.set_ylabel('AUROC')
    ax.set_ylim(0.88, 1.02)
    ax.legend()
    ax.set_title('Per-Label AUROC')

    ax          = axes[1]
    base_probs  = baseline_results['probabilities']
    lin_probs   = linear_results['probabilities']
    true_labels = baseline_results['labels']
    linestyles  = ['-', '--', ':']
    for i, (comp, comp_full, ls) in enumerate(zip(complications, COMP_FULL, linestyles)):
        fpr_b, tpr_b, _ = roc_curve(true_labels[:, i], base_probs[:, i])
        fpr_l, tpr_l, _ = roc_curve(true_labels[:, i], lin_probs[:, i])
        auc_b = baseline_results['per_label_auroc'][comp]
        auc_l = linear_results['per_label_auroc'][comp]
        ax.plot(fpr_b, tpr_b, color=C_BASE, linestyle=ls, linewidth=1.8,
                label=f'Base {comp_full[:3]} ({auc_b:.3f})')
        ax.plot(fpr_l, tpr_l, color=C_LIN,  linestyle=ls, linewidth=1.8,
                label=f'Lin  {comp_full[:3]} ({auc_l:.3f})')
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves (all labels)')
    ax.legend(fontsize=8.5, loc='lower right')

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'plot3_per_label.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("   Plot 3: Per-label AUROC + ROC curves")

    # ── PLOT 4: INTERACTION MATRIX ────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Learned Interaction Matrix A — {depth_str}',
                 fontsize=14, fontweight='bold', y=1.01)

    ax   = axes[0]
    vmax = max(abs(A_matrix.min()), abs(A_matrix.max())) + 0.005
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    im   = ax.imshow(A_matrix, cmap='RdBu_r', norm=norm, aspect='auto')
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(COMP_FULL, rotation=20, ha='right')
    ax.set_yticklabels(COMP_FULL)
    ax.set_xlabel('Source (what does the influencing)', labelpad=8)
    ax.set_ylabel('Target (what gets influenced)',      labelpad=8)
    ax.set_title('A matrix heatmap')
    ax.grid(False)
    for i in range(3):
        for j in range(3):
            val   = A_matrix[i, j]
            color = 'white' if abs(val) > vmax * 0.55 else 'black'
            ax.text(j, i, f'{val:+.4f}', ha='center', va='center',
                    fontsize=11, fontweight='bold', color=color)
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Interaction weight', fontsize=10)

    ax         = axes[1]
    pairs      = []
    values     = []
    bar_colors = []
    for i, tgt in enumerate(COMP_FULL):
        for j, src in enumerate(COMP_FULL):
            if i != j:
                pairs.append(f'{src[:3]}→{tgt[:3]}')
                values.append(A_matrix[i, j])
                bar_colors.append(C_LIN if A_matrix[i, j] >= 0 else C_BASE)
    y_pos = np.arange(len(pairs))
    ax.barh(y_pos, values, color=bar_colors, alpha=0.85, height=0.6)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pairs, fontsize=10)
    ax.set_xlabel('Interaction weight')
    ax.set_title('Off-diagonal interactions')
    for i, (val, yp) in enumerate(zip(values, y_pos)):
        xpos = val + 0.001 if val >= 0 else val - 0.001
        ha   = 'left'       if val >= 0 else 'right'
        ax.text(xpos, yp, f'{val:+.4f}', va='center', ha=ha,
                fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'plot4_A_matrix.png'),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("   Plot 4: Interaction matrix")


print(" Plotting function ready.")


In [ ]:























from google.colab import drive
drive.mount('/content/drive')

HIDDEN_DIM = 16
PROJ_DIM   = 8
N_LAYERS   = 3
BASE_DIR   = 'results/subset_linear_additive_v3/'
DRIVE_DIR  = '/content/drive/MyDrive/comorbidity_experiments/subset_linear_additive_v3/'
os.makedirs(BASE_DIR,  exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

SEEDS     = [42, 123, 456]
FRACTIONS = [1.0, 0.8, 0.6, 0.4, 0.2]

all_results = {}

for seed in SEEDS:
    all_results[seed] = {}

    for fraction in FRACTIONS:

        condition_label = f'seed{seed}_frac{int(fraction*100)}'
        condition_str   = f'Seed {seed} | {int(fraction*100)}% data'
        SAVE_DIR        = os.path.join(BASE_DIR, condition_label)
        os.makedirs(SAVE_DIR, exist_ok=True)

        print(f"\n{'='*60}")
        print(f"CONDITION: {condition_str}")
        print(f"{'='*60}")

        # ── Subsample training data ───────────────────────────────────────
        if fraction < 1.0:
            n_subset = int(len(X_train) * fraction)
            indices  = np.random.RandomState(seed).choice(
                len(X_train), n_subset, replace=False
            )
            X_tr = X_train[indices]
            y_tr = y_train[indices]
        else:
            X_tr = X_train
            y_tr = y_train

        X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
        y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
        cond_train_loader = DataLoader(
            TensorDataset(X_tr_t, y_tr_t),
            batch_size=batch_size, shuffle=True
        )
        print(f"Training samples: {len(X_tr)}")

        # ── Create backbone and snapshot init state ───────────────────────
        set_seed(seed)
        backbone   = SharedBackbone(input_dim, hidden_dim=HIDDEN_DIM,
                                    n_hidden_layers=N_LAYERS, proj_dim=PROJ_DIM)
        init_state = {k: v.cpu().clone() for k, v in backbone.state_dict().items()}

        # Save backbone init for reproducibility
        torch.save(init_state,
                   os.path.join(SAVE_DIR, 'backbone_init.pt'))

        # ── Train Baseline ────────────────────────────────────────────────
        set_seed(seed)
        baseline_model = BaselineModel(backbone)
        baseline_model, baseline_losses, baseline_aurocs, baseline_best_val = train_model(
            baseline_model, cond_train_loader, val_loader,
            f'Baseline | {condition_str}'
        )
        baseline_results = evaluate_model(baseline_model, test_loader)
        print(f"Baseline Test AUROC: {baseline_results['auroc_macro']:.4f}")

        # ── Train Linear Additive ─────────────────────────────────────────
        backbone_linear = SharedBackbone(input_dim, hidden_dim=HIDDEN_DIM,
                                         n_hidden_layers=N_LAYERS, proj_dim=PROJ_DIM)
        backbone_linear.load_state_dict(init_state)
        set_seed(seed)
        linear_model = LinearAdditiveModel(backbone_linear)
        linear_model, linear_losses, linear_aurocs, linear_best_val = train_model(
            linear_model, cond_train_loader, val_loader,
            f'LinAdd  | {condition_str}'
        )
        linear_results = evaluate_model(linear_model, test_loader)
        print(f"Linear Additive Test AUROC: {linear_results['auroc_macro']:.4f}")

        # ── Extract A matrix ──────────────────────────────────────────────
        A_matrix = linear_model.head.A.detach().cpu().numpy()
        np.fill_diagonal(A_matrix, 0)

        gap = linear_results['auroc_macro'] - baseline_results['auroc_macro']
        print(f"Gap: {gap:+.4f}")

        # ── Save plots ────────────────────────────────────────────────────
        try:
            save_all_plots(
                N_LAYERS, SAVE_DIR,
                baseline_losses, linear_losses,
                baseline_aurocs, linear_aurocs,
                baseline_best_val, linear_best_val,
                baseline_results, linear_results,
                A_matrix
            )
        except Exception as e:
            print(f"   Plot error: {e}")

        # ── Save model weights ────────────────────────────────────────────
        torch.save(baseline_model.state_dict(),
                   os.path.join(SAVE_DIR, 'baseline_model.pt'))
        torch.save(linear_model.state_dict(),
                   os.path.join(SAVE_DIR, 'linear_model.pt'))
        np.save(os.path.join(SAVE_DIR, 'A_matrix.npy'), A_matrix)

        # ── Save results JSON ─────────────────────────────────────────────
        condition_results = {
            'seed':         seed,
            'fraction':     fraction,
            'n_train':      len(X_tr),
            'condition_label': condition_label,
            'baseline': {
                'best_val_auroc':  baseline_best_val,
                'auroc_macro':     baseline_results['auroc_macro'],
                'per_label_auroc': baseline_results['per_label_auroc'],
                'f1_macro':        baseline_results['f1_macro'],
                'hamming_loss':    baseline_results['hamming_loss'],
                'subset_accuracy': baseline_results['subset_accuracy'],
                'train_losses':    baseline_losses,
                'val_aurocs':      baseline_aurocs,
            },
            'linear_additive': {
                'best_val_auroc':  linear_best_val,
                'auroc_macro':     linear_results['auroc_macro'],
                'per_label_auroc': linear_results['per_label_auroc'],
                'f1_macro':        linear_results['f1_macro'],
                'hamming_loss':    linear_results['hamming_loss'],
                'subset_accuracy': linear_results['subset_accuracy'],
                'train_losses':    linear_losses,
                'val_aurocs':      linear_aurocs,
                'A_matrix':        A_matrix.tolist(),
            },
            'gap': gap,
        }

        with open(os.path.join(SAVE_DIR, 'results.json'), 'w') as f:
            json.dump(condition_results, f, indent=2)

        all_results[seed][fraction] = condition_results

        # ── Copy to Drive ─────────────────────────────────────────────────
        drive_condition_dir = os.path.join(DRIVE_DIR, condition_label)
        os.makedirs(drive_condition_dir, exist_ok=True)
        for filename in os.listdir(SAVE_DIR):
            shutil.copy2(
                os.path.join(SAVE_DIR, filename),
                os.path.join(drive_condition_dir, filename)
            )
        print(f"   Saved to Drive: {drive_condition_dir}")


# MASTER SUMMARY

print(f"\n{'='*70}")
print("SUBSET EXPERIMENT — MASTER SUMMARY")
print(f"{'='*70}")
print(f"\n{'Fraction':<12} {'Seed':<8} {'Base AUROC':<14} {'Lin AUROC':<14} {'Δ':<10} {'Winner'}")
print("-"*70)

for seed in SEEDS:
    for fraction in FRACTIONS:
        r      = all_results[seed][fraction]
        b_auc  = r['baseline']['auroc_macro']
        l_auc  = r['linear_additive']['auroc_macro']
        gap    = r['gap']
        winner = 'Linear ' if gap > 0 else 'Baseline'
        print(f"{int(fraction*100):<12} {seed:<8} {b_auc:<14.4f} {l_auc:<14.4f} {gap:<+10.4f} {winner}")

# Aggregated by fraction
print(f"\n{'='*70}")
print("AGGREGATED (mean ± std across seeds)")
print(f"{'='*70}")
print(f"\n{'Fraction':<12} {'Train N':<10} {'Base AUROC':<20} {'Lin AUROC':<20} {'Δ':<15} {'Winner'}")
print("-"*70)

for fraction in FRACTIONS:
    b_aucs = [all_results[s][fraction]['baseline']['auroc_macro']        for s in SEEDS]
    l_aucs = [all_results[s][fraction]['linear_additive']['auroc_macro'] for s in SEEDS]
    gaps   = [all_results[s][fraction]['gap']                            for s in SEEDS]
    n_tr   = all_results[SEEDS[0]][fraction]['n_train']
    winner = 'Linear ' if np.mean(gaps) > 0 else 'Baseline'
    print(f"{int(fraction*100):<12} {n_tr:<10} "
          f"{np.mean(b_aucs):.4f} ± {np.std(b_aucs):.4f}    "
          f"{np.mean(l_aucs):.4f} ± {np.std(l_aucs):.4f}    "
          f"{np.mean(gaps):+.4f} ± {np.std(gaps):.4f}    "
          f"{winner}")

# Save master JSON
master = {
    'experiment':  'subset_linear_additive_v3',
    'model_a':     'Baseline',
    'model_b':     'Linear Additive',
    'architecture': {
        'hidden_dim':      HIDDEN_DIM,
        'proj_dim':        PROJ_DIM,
        'n_hidden_layers': N_LAYERS,
    },
    'seeds':     SEEDS,
    'fractions': FRACTIONS,
    'results':   {str(s): {str(f): v for f, v in sv.items()}
                  for s, sv in all_results.items()}
}

master_path = os.path.join(BASE_DIR, 'master_results.json')
with open(master_path, 'w') as f:
    json.dump(master, f, indent=2)

shutil.copy2(master_path, os.path.join(DRIVE_DIR, 'master_results.json'))
print(f"\n Master results saved to Drive.")
print(f"{'='*70}")
